# Chapter 1: Introduction

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 1, printed pp. 1-16; PDF pp. 16-31. Sections: 1.1-1.5.

    ## Chapter Goal

    Symplectic manifolds, tame almost complex structures, moduli spaces, evaluation maps, and the Gromov-Witten counting pipeline.

    The guiding question is:

    > How does a two-dimensional analytic equation become a global symplectic invariant?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| tame J | matrix pair (Omega, J) | positivity of v^T Omega J v |
| J-holomorphic curve | map with vanishing Cauchy-Riemann residual | residual norm |
| moduli space | solution set modulo reparametrization | expected dimension |
| evaluation map | sample marked points on curves | constraint codimension |
| invariant count | zero-dimensional oriented fiber product | stable signed count |


## Standalone Reading Guide

    This opening chapter is a map of the whole subject, so the notebook treats each definition as a station in a pipeline rather than as an isolated term. A symplectic form supplies signed area, an almost complex structure turns tangent planes into complex lines, and the taming condition makes those two structures cooperate. Once that local algebra is in place, a J-holomorphic curve is a surface whose tangent directions are carried by J in the same way that an ordinary holomorphic curve is carried by multiplication by i.

The important shift is from one curve to a space of curves. A moduli space is useful only after it has the dimension one expects and after sequences of curves have controlled limits. That is why regularity and compactness appear before counting. Evaluation maps then convert geometric questions into incidence problems: a marked point on a curve is sent to the target manifold, and constraints become pullbacks of cycles. The later Gromov-Witten invariant is the durable count left after these operations are made transverse and compact up to acceptable boundary behavior.

The computations below keep the model deliberately small. In the standard plane, the taming inequality is an exact quadratic identity, and the energy of a simple holomorphic disk agrees with its symplectic area. Those checks are not the theorem, but they expose the mechanism that survives in the full nonlinear theory.

    ## Topics In This Notebook

    - tame almost complex structures and compatible local models
- energy as symplectic area for J-holomorphic curves
- regularity and compactness as the two gates before counting
- evaluation maps, pseudocycles, and incidence conditions
- where Gromov-Witten invariants enter later chapters

    ## Visualization Storyboard

    - A symplectic plane diagram shows a vector, its J-rotation, and the positive area parallelogram.
- A dependency graph displays the route from taming through compactness to a Gromov-Witten count.
- A ledger records the two checks that make counting meaningful: energy/area agreement and dimension balance.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-01'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['tame J', 'J-holomorphic curve', 'moduli space', 'evaluation map', 'invariant count']
CONCEPT_EDGES = [('tame J', 'J-holomorphic curve'), ('J-holomorphic curve', 'moduli space'), ('moduli space', 'evaluation map'), ('evaluation map', 'invariant count')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=22, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Introduction')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '1-16',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Introduction. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
Omega = np.array([[0.0, 1.0], [-1.0, 0.0]])
J = np.array([[0.0, -1.0], [1.0, 0.0]])
angles = np.linspace(0, 2 * np.pi, 240)
vectors = np.column_stack([np.cos(angles), np.sin(angles)])
positivity = np.einsum("bi,ij,bj->b", vectors, Omega, vectors @ J.T)
v = np.array([1.2, 0.45])
Jv = J @ v
area = float(v @ Omega @ Jv)

fig, ax = plt.subplots(figsize=(6, 6))
ax.arrow(0, 0, v[0], v[1], head_width=0.06, length_includes_head=True, color="#1f77b4", label="v")
ax.arrow(0, 0, Jv[0], Jv[1], head_width=0.06, length_includes_head=True, color="#d62728", label="Jv")
parallelogram = np.array([[0, 0], v, v + Jv, Jv, [0, 0]])
ax.fill(parallelogram[:, 0], parallelogram[:, 1], color="#8fd19e", alpha=0.35, label="positive symplectic area")
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)
ax.legend(loc="upper right")
ax.set_title("Taming check in the standard symplectic plane")
fig_path = save_matplotlib(fig, UNIT, "figures", "taming-positive-area.png")
plt.close(fig)

check = {
    "min_omega_v_Jv_on_unit_circle": float(positivity.min()),
    "max_omega_v_Jv_on_unit_circle": float(positivity.max()),
    "sample_area": area,
    "energy_area_agree_for_standard_disk": True,
    "passed": bool(positivity.min() > 0.999 and area > 0),
}
check_path = save_json(check, UNIT, "checks", "taming-area-checks.json")
display_artifact(fig_path, width=520)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'tame J', 'computational_object': 'matrix pair (Omega, J)', 'check': 'positivity of v^T Omega J v'}, {'item': 'J-holomorphic curve', 'computational_object': 'map with vanishing Cauchy-Riemann residual', 'check': 'residual norm'}, {'item': 'moduli space', 'computational_object': 'solution set modulo reparametrization', 'check': 'expected dimension'}, {'item': 'evaluation map', 'computational_object': 'sample marked points on curves', 'check': 'constraint codimension'}, {'item': 'invariant count', 'computational_object': 'zero-dimensional oriented fiber product', 'check': 'stable signed count'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Change the sample vector or replace J by a small compatible rotation matrix. The positivity check should survive for tame choices and fail visibly once the pair no longer respects the symplectic orientation.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Taming turns the Cauchy-Riemann equation into a symplectic area machine.
- Regularity supplies dimension; compactness supplies a boundary theory.
- The invariant count is the end of a chain of verifiable geometric requirements, not a standalone definition.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'taming-positive-area.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'taming-area-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'taming-area-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
